# ECEN743 Spring 2026 - Assignment 4
## Policy Gradient

General Instructions  

1. This code consists of TODO blocks, read them carefully and complete each of the blocks
2. Type your code between the following lines  
        ###### TYPE YOUR CODE HERE ######  
        ###############################
3. The default hyperparameters should be able to solve LunarLander-v3
4. You do not need to modify the rest of the code for this assignment.

Run the following initializer. Make sure the path points to our rl_env.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import os
import math
import torch
import random
import argparse
import numpy as np
import torch.nn as nn
from tqdm import trange
import gymnasium as gym
from pprint import pprint
import torch.optim as optim
from collections import deque
import torch.nn.functional as F
import matplotlib.pyplot as plt
from gymnasium.wrappers import RecordVideo

In [ ]:
class ValueNetwork(nn.Module):
    """A neural network that estimates the value function of a state.
    
    This network acts as the baseline in Policy Gradient algorithms to reduce 
    variance during training by approximating the expected return from a given state.
    """
    def __init__(self, state_dim):
        """Initializes the ValueNetwork.
        
        Args:
            state_dim (int): The dimensionality of the state space.
        """
        super(ValueNetwork, self).__init__()
        self.l1 = nn.Linear(state_dim, 64)
        self.l2 = nn.Linear(64, 64)
        self.l3 = nn.Linear(64, 1)

    def forward(self, state):
        """Computes the estimated value for a given state observation.
        
        Args:
            state (torch.Tensor): A tensor representing the environment state(s).
                Shape: (batch_size, state_dim).
                
        Returns:
            torch.Tensor: The estimated state value. Shape: (batch_size, 1).
        """
        v = torch.tanh(self.l1(state))
        v = torch.tanh(self.l2(v))
        return self.l3(v)

In [ ]:
class PolicyNetwork(nn.Module):
    """A stochastic continuous policy network parameterized by a Gaussian distribution.
    
    Given a state, this network outputs the mean of the action distribution. 
    The standard deviation is maintained as a standalone, state-independent learnable parameter.
    """
    def __init__(self, state_dim, action_dim, log_std=0.0):
        """Initializes the PolicyNetwork.
        
        Args:
            state_dim (int): The dimensionality of the state space.
            action_dim (int): The dimensionality of the continuous action space.
            log_std (float, optional): The initial log standard deviation for the 
                action distribution. Defaults to 0.0.
        """
        super(PolicyNetwork, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.l1 = nn.Linear(state_dim, 64)
        self.l2 = nn.Linear(64, 64)
        self.mean = nn.Linear(64, action_dim)
        
        # The log standard deviation is a learnable parameter independent of the state
        self.log_std = nn.Parameter(torch.ones(1, action_dim) * log_std)

    def forward(self, state):
        """Passes the state through the network to determine the action distribution parameters.
        
        Args:
            state (torch.Tensor): The environment state(s). Shape: (batch_size, state_dim).
            
        Returns:
            tuple:
                - a_mean (torch.Tensor): The mean of the action distribution. Shape: (batch_size, action_dim).
                - a_log_std (torch.Tensor): The log standard deviation. Shape: (batch_size, action_dim).
                - a_std (torch.Tensor): The standard deviation. Shape: (batch_size, action_dim).
        """
        a = torch.tanh(self.l1(state))
        a = torch.tanh(self.l2(a))
        a_mean = self.mean(a)
        a_log_std = self.log_std.expand_as(a_mean)
        a_std = torch.exp(a_log_std)        
        return a_mean, a_log_std, a_std

    def select_action(self, state):
        """Samples an action from the policy's predicted Gaussian distribution.
        
        Args:
            state (torch.Tensor): The environment state(s). Shape: (batch_size, state_dim).
            
        Returns:
            torch.Tensor: An action sampled from the Normal(mean, std) distribution.
                Shape: (batch_size, action_dim).
        """
        a_mean, _, a_std = self.forward(state)
        action = torch.normal(a_mean, a_std)
        return action
    
    def get_log_prob(self, state, action):
        """Calculates the log probability density of taking a specific action in a specific state.
        
        Args:
            state (torch.Tensor): The environment state(s). Shape: (batch_size, state_dim).
            action (torch.Tensor): The action(s) taken. Shape: (batch_size, action_dim).
            
        Returns:
            torch.Tensor: The sum of log probabilities for the given action(s). 
                Shape: (batch_size, 1).
        """
        mean, log_std, std = self.forward(state)
        var = std.pow(2)
        log_density = -(action - mean).pow(2) / (2 * var) - 0.5 * math.log(2 * math.pi) - log_std
        return log_density.sum(1, keepdim=True)

In [ ]:
class PGAgent():
    """An agent that handles training loop interactions and Policy Gradient updates."""
    
    def __init__(self, state_dim, action_dim, discount=0.99, lr=1e-3, gpu_index=0, seed=0, env="LunarLander-v3", **kwargs):
        """Initializes the Policy Gradient Agent.
        
        Args:
            state_dim (int): Dimensionality of the state space.
            action_dim (int): Dimensionality of the action space.
            discount (float, optional): The discount factor (gamma) for future rewards. Defaults to 0.99.
            lr (float, optional): Learning rate for both policy and value optimizers. Defaults to 1e-3.
            gpu_index (int, optional): The index of the GPU to use for training. Defaults to 0.
            seed (int, optional): Random seed for reproducibility. Defaults to 0.
            env (str, optional): The name of the Gymnasium environment. Defaults to "LunarLander-v3".
            **kwargs: Additional keyword arguments.
        """
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.discount = discount
        self.lr = lr
        self.device = torch.device(f'cuda:{gpu_index}' if torch.cuda.is_available() else 'cpu')
        self.env_name = env
        self.seed = seed
        
        self.policy = PolicyNetwork(state_dim, action_dim)
        self.value = ValueNetwork(state_dim)
        self.optimizer_policy = torch.optim.Adam(self.policy.parameters(), lr=self.lr)
        self.optimizer_value = torch.optim.Adam(self.value.parameters(), lr=self.lr)

    def sample_traj(self, batch_size=2000, evaluate=False):
        """Samples trajectories from the environment using the current policy.
        
        Args:
            batch_size (int, optional): Minimum number of environment steps to collect 
                before returning. Defaults to 2000.
            evaluate (bool, optional): If True, the agent acts deterministically (uses the mean 
                action) and only returns the episodic reward. Defaults to False.
                
        Returns:
            If evaluate=True:
                float: The mean episodic reward of the collected trajectories.
            If evaluate=False:
                tuple: A tuple containing lists of states, actions, rewards, not_dones 
                (0 if done, 1 otherwise), and the mean episodic reward.
        """
        self.policy.to("cpu")
        env = gym.make(self.env_name, continuous=True)
        states, actions, rewards, n_dones = [], [], [], []
        curr_reward_list = []
        
        while len(states) < batch_size:
            state, _ = env.reset(seed=self.seed)
            curr_reward = 0
            for t in range(1000):
                state_ten = torch.from_numpy(state).float().unsqueeze(0)
                with torch.no_grad():
                    if evaluate:
                        action = self.policy(state_ten)[0][0].numpy()
                    else:
                        action = self.policy.select_action(state_ten)[0].numpy()
                
                action = action.astype(np.float64)
                n_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                
                states.append(state)
                actions.append(action)
                rewards.append(reward)
                n_done = 0 if done else 1
                n_dones.append(n_done)
                
                state = n_state
                curr_reward += reward
                if done:
                    break
            curr_reward_list.append(curr_reward)
            
        env.close()
        
        if evaluate:
            return np.mean(curr_reward_list)
        return states, actions, rewards, n_dones, np.mean(curr_reward_list)
    
    def update(self, states, actions, rewards, n_dones, update_type='Baseline'):
        """Computes losses and updates the network parameters based on the chosen PG algorithm.
        
        Args:
            states (list): List of state arrays collected during `sample_traj`.
            actions (list): List of action arrays collected during `sample_traj`.
            rewards (list): List of scalar rewards collected during `sample_traj`.
            n_dones (list): List of inverse terminal flags (1 if not done, 0 if done).
            update_type (str, optional): The specific PG algorithm variant to run. 
                Must be one of ['Rt', 'Gt', 'Baseline']. Defaults to 'Baseline'.
        """
        self.policy.to(self.device)
        if update_type == "Baseline":
            self.value.to(self.device)
            
        states_ten = torch.from_numpy(np.stack(states)).to(self.device)
        action_ten = torch.from_numpy(np.stack(actions)).to(self.device)
        rewards_ten = torch.from_numpy(np.stack(rewards)).to(self.device)
        n_dones_ten = torch.from_numpy(np.stack(n_dones)).to(self.device)

        if update_type == "Rt":
            """
            TODO: Perform PG using the cumulative discounted reward of the entire trajectory.
            1. Compute the discounted reward of each trajectory (rt).
            2. Compute log probabilities using states_ten and action_ten.
            3. Compute policy loss and update the policy.
            """
            ###### TYPE YOUR CODE HERE ######
            #################################

        if update_type == 'Gt':
            """
            TODO: Perform PG using reward_to_go.
            1. Compute reward_to_go (gt) using rewards_ten and n_dones_ten.
            2. gt should be of the same length as rewards_ten.
            3. Compute log probabilities using states_ten and action_ten.
            4. Compute policy loss and update the policy.
            """
            gt = torch.zeros(rewards_ten.shape[0], 1).to(self.device)

            ###### TYPE YOUR CODE HERE ######
            # Do steps 1-2
            #################################

            gt = (gt - gt.mean()) / (gt.std() + 1e-8)

            ###### TYPE YOUR CODE HERE ######
            # Do steps 3-4
            #################################

        if update_type == 'Baseline':
            """
            TODO: Perform PG using reward_to_go and baseline.
            1. Compute values of states, this will be used as the baseline.
            2. Compute reward_to_go (gt) using rewards_ten and n_dones_ten.
            3. gt should be of the same length as rewards_ten.
            4. Compute advantages.
            5. Update the value network to predict gt for each state (L2 norm).
            6. Compute log probabilities using states_ten and action_ten.
            7. Compute policy loss (using advantages) and update the policy.
            """
            with torch.no_grad():
                values_adv = self.value(states_ten)
            gt = torch.zeros(rewards_ten.shape[0], 1).to(self.device)

            ###### TYPE YOUR CODE HERE ######
            # Do steps 1-4
            #################################

            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            ###### TYPE YOUR CODE HERE ######
            # Do steps 5-7
            #################################

In [ ]:
def train_loop(args):
    """Executes the main training loop for the Policy Gradient agent.
    
    Args:
        args (argparse.Namespace): A namespace object containing training hyperparameters.
        
    Returns:
        tuple: A tuple containing the trained PGAgent instance and a list of evaluation rewards 
            collected throughout training.
    """
    print('\n\nTraining')
    
    env = gym.make(args.env, continuous=True)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    env.close()

    agent = PGAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        **vars(args)
    )

    moving_window = deque(maxlen=10)
    eval_rewards = []
    
    episode_pbar = trange(args.n_iter)
    for e in episode_pbar:
        """
        Steps of PG algorithm:
        1. Sample environment to gather data using a policy
        2. Update the policy using the data
        3. Evaluate the updated policy
        4. Repeat 1-3
        """
        states, actions, rewards, n_dones, train_reward = agent.sample_traj(batch_size=args.batch_size)
        agent.update(states, actions, rewards, n_dones, args.algo)
        eval_reward = agent.sample_traj(evaluate=True)
        
        moving_window.append(eval_reward)
        eval_rewards.append(eval_reward)
        
        episode_pbar.set_description(f'Train Rwd: {train_reward:8.2f} | Eval Rwd: {eval_reward:8.2f} | Avg Eval Rwd: {np.mean(moving_window):8.2f}')

    return agent, eval_rewards

In [ ]:
def plot_rewards(eval_rewards, run_name):
    """Generates and saves a plot of the evaluation rewards over training iterations.
    
    Args:
        eval_rewards (list): List of evaluation rewards collected during training.
        run_name (str): The name of the current run, used for titling and saving the plot.
        
    TODO:
        1. Compute a smoothed version of the evaluation rewards using a moving average window of size 10.
        2. Plot the original episodic evaluation rewards.
        3. Overlay the smoothed reward curve to visualize training progress more clearly.
        4. Label the plot with appropriate titles and axis names.
        5. Save the plot.
    """
    ###### TYPE YOUR CODE HERE ######
    #################################

In [ ]:
def test_loop(agent, env_name, run_name):
    """Runs a visual evaluation of the trained agent and saves a video recording.
    
    Args:
        agent (PGAgent): The fully trained Policy Gradient agent.
        env_name (str): The name of the Gymnasium environment.
        run_name (str): The name of the run, used to name the output video file.
        
    TODO:
        1. Create a video recording of the agent's performance during testing.
        2. Select actions using the agent's policy in evaluation mode (mean action).
    """
    print('\n\nTesting')
    env = gym.make(env_name, render_mode="rgb_array", continuous=True)
    
    ###### TYPE YOUR CODE HERE ######
    #################################

In [ ]:
args = argparse.Namespace(
    run_name="pg",
    env="LunarLander-v3",         # Gymnasium environment name
    seed=0,                       # Sets Gym, PyTorch and Numpy seeds
    n_iter=200,                   # Maximum number of training iterations
    discount=0.99,                # Discount factor
    batch_size=5000,              # Training samples in each batch of training
    lr=5e-3,                      # Learning rate
    gpu_index=0,                  # GPU index
    algo="Baseline"               # PG algorithm type. Baseline/Gt/Rt
)

pprint(vars(args), indent=4, width=2)
os.makedirs(args.run_name, exist_ok=True)

# Setting seeds
torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

# Training agent
agent, eval_rewards = train_loop(args)

# Plotting train rewards
plot_rewards(eval_rewards, args.run_name)

# Testing agent
test_loop(agent, args.env, args.run_name)